In [1]:
%config Completer.use_jedi = False
%load_ext autoreload

# Imports

In [39]:
import os
os.getpid()

818365

In [40]:
con.close()

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import logging
import math
from time import sleep, time
from pathlib import Path
import argparse
import bs4
import duckdb
import csv
import numpy as np

In [3]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("../../logs/get_ratings.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

In [4]:
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

# Functions

In [40]:
def get_boardgame_ratings(
    boardgame_data: pd.DataFrame,
    boardgame_ratings: pd.DataFrame = None,
    batch_saves: bool = False,
    batch_size: int = 20,
    log_level: str = "INFO",
    keep_partial_ratings: bool = False,
    update_numratings: bool = False,
):
    """
    Fetch user ratings for each board game from BGG API.
    The BGG API has a limit of 20 IDs per request, so we process in batches.

    Args:
        boardgame_data (pd.DataFrame): DataFrame from get_boardgame_data()
        boardgame_ratings (pd.DataFrame, optional): Existing ratings data to update
        batch_saves (bool): Whether to save data after each batch
        batch_size (int): Number of games to process in each batch. BGG API has a limit of 20 IDs per request.
        log_level (str): Logging level for this function
        drop_partial_ratings (bool): Whether to drop games with partial ratings
        update_numratings (bool): Whether to update number of ratings for games with missing ratings

    Returns:
        pd.DataFrame: DataFrame containing user ratings
    """
    # Set logging level for this function
    current_level = logger.level
    logger.setLevel(getattr(logging, log_level.upper()))

    query_time = int(time())
    data_dir = Path(__file__).parent.parent.parent / "data" / "crawler"
    data_dir.mkdir(parents=True, exist_ok=True)
    save_path = data_dir / f"boardgame_ratings_{query_time}.parquet"

    # Initialize DuckDB persistent store for ratings
    duckdb_path = data_dir / "ratings.duckdb"
    con = duckdb.connect(str(duckdb_path))
    con.execute(
        """
        CREATE TABLE IF NOT EXISTS boardgame_ratings (
            game_id BIGINT,
            rating_round DOUBLE,
            username TEXT
        );
        """
    )
    # Index to speed up de-dup checks
    con.execute(
        "CREATE INDEX IF NOT EXISTS idx_boardgame_ratings ON boardgame_ratings(game_id, rating_round, username);"
    )

    boardgame_master_dict = {}
    boardgame_data_ratings = boardgame_data.loc[boardgame_data["numratings"] > 100].sort_values(
        by="numratings", ascending=True
    )
    boardgame_ids = boardgame_data_ratings["id"].tolist()
    # Check if there are any ids which have not had all their ratings pulled down yet
    if boardgame_ratings is not None:
        df_ratings_len = boardgame_ratings.copy()
        df_ratings_len = df_ratings_len.drop(columns=["id"])
        df_ratings_len = df_ratings_len.fillna("")
        for col in df_ratings_len.columns:
            df_ratings_len[col] = df_ratings_len[col].apply(len)
        df_ratings_pulled = pd.DataFrame(
            {
                "id": boardgame_ratings["id"].tolist(),
                "ratings_pulled": df_ratings_len.sum(axis=1).tolist(),
            }
        )
        boardgame_data_ratings = boardgame_data_ratings.merge(df_ratings_pulled, on="id", how="left")
        completed_ids = boardgame_data_ratings.loc[
            (boardgame_data_ratings["ratings_pulled"] - boardgame_data_ratings["numratings"])
            / (boardgame_data_ratings["numratings"])
            >= -0.1,
            "id",
        ].tolist()
        logger.info(
            f"Found {len(completed_ids)} boardgames with all ratings already pulled to completion"
        )
        boardgame_ids = list(set(boardgame_ids).difference(set(completed_ids)))
        # reorder boardgame_ids to match boardgame_data_ratings
        boardgame_ids = boardgame_data_ratings.loc[boardgame_data_ratings["id"].isin(boardgame_ids), "id"].tolist()
        df_missing_ratings = boardgame_data_ratings.loc[
            (boardgame_data_ratings["ratings_pulled"] - boardgame_data_ratings["numratings"])
            / (boardgame_data_ratings["numratings"])
            < -0.1
        ]
        logger.info(
            f"Found {df_missing_ratings.shape[0]} boardgames with missing ratings"
        )
        if not keep_partial_ratings:
            logger.info("Dropping partial ratings")
            boardgame_ratings = boardgame_ratings.loc[
                ~(boardgame_ratings["id"].isin(df_missing_ratings["id"]))
            ]
            # Also remove any partial rows for these games from the DuckDB snapshot
            # so interim exports built from DuckDB cannot include half-complete data
            if df_missing_ratings.shape[0] > 0:
                ids_to_drop = (
                    df_missing_ratings["id"].dropna().astype("int64").tolist()
                )
                if len(ids_to_drop) > 0:
                    try:
                        con.register(
                            "to_delete_games", pd.DataFrame({"game_id": ids_to_drop})
                        )
                        con.execute(
                            """
                            DELETE FROM boardgame_ratings
                            WHERE game_id IN (SELECT game_id FROM to_delete_games);
                            """
                        )
                        con.unregister("to_delete_games")
                        logger.info(
                            f"Removed {len(ids_to_drop)} game(s) with partial ratings from DuckDB"
                        )
                    except Exception as e:
                        logger.warning(
                            f"Failed to delete partial ratings from DuckDB: {str(e)}"
                        )
        else:
            logger.info(
                "Keeping partial ratings and will continue to pull down the missing ratings"
            )

        df_ratings_tmp = boardgame_ratings.copy().set_index("id")
        df_ratings_tmp.index.name = None
        boardgame_master_dict = df_ratings_tmp.to_dict(orient="index")

        if keep_partial_ratings and df_missing_ratings.shape[0] > 0:
            for _, row in df_missing_ratings.iterrows():
                ratings_count_dict = {row["id"]: row["numratings"]}
                max_ratings_page = math.ceil(row["numratings"] / 100)
                start_page = int(row["ratings_pulled"] / 100) + 1
                boardgame_master_dict = iterate_through_ratings_pages(
                    boardgame_master_dict=boardgame_master_dict,
                    max_ratings_page=max_ratings_page,
                    ratings_count_dict=ratings_count_dict,
                    start_page=start_page,
                    batch_saves=batch_saves,
                    save_path=save_path,
                    duckdb_conn=con,
                )
                logger.info(f"Successfully completed fetching ratings for {row['id']}")
            df_ratings = (
                pd.DataFrame()
                .from_dict(data=boardgame_master_dict, orient="index")
                .reset_index(names="id")
            )
            boardgame_ids = list(
                set(boardgame_ids).difference(set(df_ratings["id"].tolist()))
            )

        # Update numratings if requested
        if update_numratings:
            game_data_save_path = data_dir / f"boardgame_data_{query_time}.parquet"
            logger.info("Updating number of ratings for games with missing ratings")
            for batch_num in range(math.ceil(len(boardgame_ids) / batch_size)):
                logger.info(
                    f"Processing numratings batch {batch_num} of {math.ceil(len(boardgame_ids) / batch_size)}"
                )
                batch_ids = boardgame_ids[
                    batch_num * batch_size : (batch_num + 1) * batch_size
                ]
                batch_ids = [str(x) for x in batch_ids]
                logger.debug(f"Processing boardgame IDs for numratings: {batch_ids}")

                bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&ratingcomments=1&id={','.join(batch_ids)}"
                bgg_response = requests.get(bg_info_url)
                soup_xml = BeautifulSoup(bgg_response.content, "xml")
                games_xml_list = soup_xml.find_all(
                    "item", attrs={"type": ["boardgame", "boardgameexpansion"]}
                )

                for game_xml in games_xml_list:
                    game_id = int(game_xml["id"])
                    if game_xml.find("comments") is not None:
                        boardgame_data.loc[
                            boardgame_data["id"] == game_id, "numratings"
                        ] = int(game_xml.find("comments")["totalitems"])
                    else:
                        boardgame_data.loc[
                            boardgame_data["id"] == game_id, "numratings"
                        ] = 0
                if batch_saves and (batch_num + 1) % 20 == 0:
                    logger.info(f"Saving batch {batch_num} data")
                    boardgame_data.to_parquet(game_data_save_path)
                    logger.info(f"Saved batch {batch_num} data to {game_data_save_path}")

                sleep(1)

            # Save updated game data            
            boardgame_data.to_parquet(game_data_save_path)
            logger.info(f"Saved updated game data to {game_data_save_path}")
            boardgame_data_ratings = boardgame_data.loc[boardgame_data["numratings"] > 100].sort_values(
                by="numratings", ascending=False
            )
            

    logger.info(f"Starting to fetch ratings for {len(boardgame_ids)} boardgames")

    for batch_num in range(math.ceil(len(boardgame_ids) / batch_size)):
        logger.info(
            f"Processing batch {batch_num + 1} of {math.ceil(len(boardgame_ids) / batch_size)}"
        )
        batch_ids = boardgame_ids[batch_num * batch_size : (batch_num + 1) * batch_size]
        df_batch_games = boardgame_data_ratings.loc[boardgame_data_ratings["id"].isin(batch_ids)]
        ratings_count_dict = pd.Series(
            df_batch_games["numratings"].values,
            index=df_batch_games["id"],
        ).to_dict()
        max_ratings_page = math.ceil(max(ratings_count_dict.values()) / 100)
        logger.info(
            f"Processing {max_ratings_page} rating pages for batch {batch_num + 1}"
        )
        boardgame_master_dict = iterate_through_ratings_pages(
            boardgame_master_dict=boardgame_master_dict,
            max_ratings_page=max_ratings_page,
            ratings_count_dict=ratings_count_dict,
            batch_saves=batch_saves,
            save_path=save_path,
            duckdb_conn=con,
        )

        # Ratings are persisted to DuckDB incrementally; skip interim Parquet writes

    if len(boardgame_ids) > 0:
        logger.info("Successfully completed fetching all ratings. Exporting snapshot to Parquet...")
        df_ratings_wide = build_wide_ratings_df_from_duckdb(con)
        df_ratings_wide.to_parquet(save_path)
        logger.info(f"Saved final data to {save_path}")
    else:
        logger.warning("No ratings were fetched")

    # Restore original logging level
    logger.setLevel(current_level)
    con.close()


def iterate_through_ratings_pages(
    boardgame_master_dict: dict,
    max_ratings_page: int,
    ratings_count_dict: dict,
    start_page: int = 1,
    batch_saves: bool = False,
    save_path: str = None,
    duckdb_conn: duckdb.DuckDBPyConnection = None,
):
    """
    Helper function to iterate through paginated rating data from BGG API.

    Args:
        boardgame_master_dict (dict): Dictionary to store rating data
        max_ratings_page (int): Maximum number of rating pages to process. Derived from the number of ratings for each game.
        ratings_count_dict (dict): Dictionary mapping game IDs to number of ratings
        start_page (int): Page number to start processing from
        batch_saves (bool): Whether to save data periodically
        save_path (str): Path to save data to

    Returns:
        dict: Updated dictionary containing rating data
    """
    for page_num in range(start_page, max_ratings_page + 1):
        # Only grab the pages for games which have enough ratings to be on the page num
        batch_ids_ratings = [
            str(x)
            for x in ratings_count_dict.keys()
            if math.ceil(ratings_count_dict[x] / 100) >= page_num
        ]
        bg_rating_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&ratingcomments=1&pagesize=100&page={page_num}&id={','.join(batch_ids_ratings)}"
        bgg_rating_response = requests.get(bg_rating_url)
        soup_rating_xml = BeautifulSoup(bgg_rating_response.content, "xml")
        ratings_xml_list = soup_rating_xml.find_all("item", attrs={"type": "boardgame"})

        rows_for_page = []
        for game_xml in ratings_xml_list:
            game_id = int(game_xml["id"])
            if game_id not in boardgame_master_dict:
                boardgame_master_dict[game_id] = {}
            per_game_new = parse_ratings_to_dict(game_xml)
            for rating_round, users in per_game_new.items():
                if rating_round not in boardgame_master_dict[game_id]:
                    boardgame_master_dict[game_id][rating_round] = list(users)
                else:
                    boardgame_master_dict[game_id][rating_round].extend(users)
            for rating_round, users in per_game_new.items():
                for username in users:
                    rows_for_page.append((game_id, float(rating_round), username))

        if duckdb_conn is not None and len(rows_for_page) > 0:
            df_insert = pd.DataFrame(rows_for_page, columns=["game_id", "rating_round", "username"])
            df_insert = df_insert.drop_duplicates()
            duckdb_conn.register("ratings_tmp", df_insert)
            duckdb_conn.execute(
                """
                INSERT INTO boardgame_ratings
                SELECT game_id, rating_round, username
                FROM ratings_tmp t
                WHERE NOT EXISTS (
                    SELECT 1 FROM boardgame_ratings b
                    WHERE b.game_id = t.game_id
                      AND b.rating_round = t.rating_round
                      AND b.username = t.username
                );
                """
            )
            duckdb_conn.unregister("ratings_tmp")

        if page_num % 100 == 0:
            logger.info(f"Processed ratings page {page_num} of {max_ratings_page}")
        sleep(1)
    return boardgame_master_dict


def parse_ratings_to_dict(game_xml: bs4.element.Tag) -> dict:
    """
    Parse a game's XML into {rating_round_str: [usernames...]}
    without mutating an existing aggregate.
    """
    game_dict: dict = {}
    ratings_list = game_xml.find_all("comment")
    for rating in ratings_list:
        rating_round = str(round(2 * float(rating["rating"])) / 2)
        if rating_round not in game_dict:
            game_dict[rating_round] = [rating["username"]]
        else:
            game_dict[rating_round].append(rating["username"])
    return game_dict


def build_wide_ratings_df_from_duckdb(con: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    """
    Build the wide ratings DataFrame (id + rating bucket columns of username lists)
    from the persistent DuckDB table to match prior Parquet format.
    """
    df_long = con.execute(
        """
        SELECT game_id, rating_round, list(username) AS usernames
        FROM boardgame_ratings
        GROUP BY game_id, rating_round
        """
    ).fetch_df()

    df_long["usernames"] = df_long["usernames"].apply(lambda x: x.tolist())
    df_wide = df_long.pivot(index="game_id", columns="rating_round", values="usernames")
    df_wide.columns.name = None
    df_wide = df_wide.reset_index(names="id")
    df_wide.columns = df_wide.columns.astype(str)

    # master: dict = {}
    # for _, row in df_long.iterrows():
    #     game_id = int(row["game_id"])
    #     rating_key = str(row["rating_round"])  # keep columns as str like prior format
    #     usernames = row["usernames"]
    #     # DuckDB may return a list type for list_agg
    #     if isinstance(usernames, list):
    #         usernames_list = usernames
    #     else:
    #         usernames_list = [usernames]
    #     if game_id not in master:
    #         master[game_id] = {}
    #     master[game_id][rating_key] = usernames_list

    # if len(master) == 0:
    #     return pd.DataFrame(columns=["id"])  # empty

    # df_wide = (
    #     pd.DataFrame()
    #     .from_dict(data=master, orient="index")
    #     .reset_index(names="id")
    # )
    return df_wide

# Code

In [6]:
# Get the most recent game_ranks file
notebook_dir = Path.cwd()
data_dir = notebook_dir.parent.parent / "data" / "crawler"
game_ranks_files = list(data_dir.glob("boardgame_ranks_*.csv"))
if not game_ranks_files:
    raise FileNotFoundError("No game ranks files found")
latest_ranks = max(game_ranks_files, key=lambda x: x.stat().st_mtime)
logger.info(f"Using game ranks file: {latest_ranks}")
df_ranks = pd.read_csv(latest_ranks, sep="|", escapechar="\\", quoting=csv.QUOTE_NONE, usecols=["id", "is_expansion"])
non_expansion_ids = df_ranks.loc[df_ranks["is_expansion"] == 0, "id"].tolist()

2025-09-17 15:16:25,348 - __main__ - INFO - Using game ranks file: /home/ubuntu/git/pax_tt_recommender/data/crawler/boardgame_ranks_20250910.csv


In [7]:
notebook_dir = Path.cwd()
data_dir = notebook_dir.parent.parent / "data" / "crawler"
game_files = list(data_dir.glob("boardgame_data_*.parquet"))
if not game_files:
    raise FileNotFoundError("No game data files found")

In [8]:
latest_games = max(game_files, key=lambda x: x.stat().st_mtime)
logger.info(f"Using game data file: {latest_games}")

2025-09-17 15:16:28,545 - __main__ - INFO - Using game data file: /home/ubuntu/git/pax_tt_recommender/data/crawler/boardgame_data_1757622241.parquet


In [10]:
# Read game data
df_games = pd.read_parquet(latest_games)
df_games = df_games.loc[df_games["id"].isin(non_expansion_ids)]
df_games.head()

,id,thumbnail,image,description,minplayers,maxplayers,playingtime,minplaytime,maxplaytime,minage,boardgamecategory,boardgamemechanic,boardgamefamily,boardgameexpansion,boardgameartist,boardgamecompilation,boardgameimplementation,boardgamedesigner,boardgamepublisher,boardgameintegration,stddev,median,averageweight,owned,trading,wanting,wishing,numcomments,numweights,numratings,suggested_playerage,suggested_playerage_quartiles,language_dependence,language_dependence_quartiles,player_count_recs,suggested_numplayers,versions
0,224517,https://cf.geekdo-images.com/x3zxjr-Vw5iU4yDPg...,https://cf.geekdo-images.com/x3zxjr-Vw5iU4yDPg...,Brass: Birmingham is an economic strategy game...,2,4,120,60,120,14,"{'2726': 'Age of Reason', '1021': 'Economic', ...","{'2956': 'Chaining', '2875': 'End Game Bonuses...","{'17519': 'Cities: Birmingham (England)', '816...",{},"{'32887': 'Gavan Brown', '70571': 'Lina Cosset...",{},"{'452264': 'Brass: Pittsburgh', '28720': 'Bras...","{'32887': 'Gavan Brown', '32943': 'Matt Tolman...","{'21765': 'Roxley', '3475': 'Arclight Games', ...",{},1.42121,0.0,3.8706,76735,283,1753,20434,7402,2619,54185,"{'total_votes': 184, '2': 1, '3': 0, '4': 0, '...","{'25 percent': '12', '75 percent': '14', '50 p...","{'total_votes': 55, '1': 52, '2': 3, '3': 0, '...","{'75 percent': 'No necessary in-game text', '5...","{'Not Recommended': ['1', '4+'], 'Recommended'...","{'total_votes': 1298, '1': {'Best': 0, 'Recomm...","[{'version_id': 523778, 'width': 12, 'length':..."
1,161936,https://cf.geekdo-images.com/-Qer2BBPG7qGGDu6K...,https://cf.geekdo-images.com/-Qer2BBPG7qGGDu6K...,Pandemic Legacy is a co-operative campaign gam...,2,4,60,60,60,13,"{'1084': 'Environmental', '2145': 'Medical'}","{'2001': 'Action Points', '2023': 'Cooperative...","{'64952': 'Components: Map (Global Scale)', '6...",{},{'14057': 'Chris Quilliams'},{},"{'221107': 'Pandemic Legacy: Season 2', '30549...","{'442': 'Rob Daviau', '378': 'Matt Leacock'}","{'538': 'Z-Man Games', '15889': 'Asterion Pres...",{},1.60501,0.0,2.8285,87503,513,810,14518,8404,1528,56640,"{'total_votes': 203, '2': 1, '3': 1, '4': 0, '...","{'25 percent': '10', '75 percent': '12', '50 p...","{'total_votes': 95, '6': 1, '7': 0, '8': 9, '9...",{'75 percent': 'Extensive use of text - massiv...,"{'Not Recommended': ['1', '4+'], 'Recommended'...","{'total_votes': 894, '1': {'Best': 21, 'Recomm...","[{'version_id': 329714, 'width': 9, 'length': ..."
2,342942,https://cf.geekdo-images.com/SoU8p28Sk1s8MSvoM...,https://cf.geekdo-images.com/SoU8p28Sk1s8MSvoM...,"In Ark Nova, you will plan and design a modern...",1,4,150,90,150,14,"{'1089': 'Animals', '1002': 'Card Game', '1084...","{'2912': 'Contracts', '2875': 'End Game Bonuse...","{'110926': 'Animals: Okapi', '67874': 'Compone...",{'452698': 'Ark Nova: 3D Colored Arcade Promo ...,"{'138547': 'Steffen Bieker', '11462': 'Loïc Bi...",{},{'441696': 'Sanctuary'},{'138517': 'Mathias Wigge'},"{'22380': 'Feuerland Spiele', '30958': 'Capsto...",{},1.39170,0.0,3.7864,80077,421,1074,16253,7266,2851,55120,"{'total_votes': 299, '2': 1, '3': 0, '4': 0, '...","{'50 percent': '12', '25 percent': '12', '75 p...","{'total_votes': 74, '11': 0, '12': 0, '13': 25...",{'25 percent': 'Moderate in-game text - needs ...,"{'Recommended': ['1', '3'], 'Best': ['2'], 'No...","{'total_votes': 2156, '1': {'Best': 210, 'Reco...","[{'version_id': 623699, 'width': 12, 'length':..."
3,174430,https://cf.geekdo-images.com/sZYp_3BTDGjh2unaZ...,https://cf.geekdo-images.com/sZYp_3BTDGjh2unaZ...,Gloomhaven is a game of Euro-inspired tactica...,1,4,120,60,120,14,"{'1022': 'Adventure', '1020': 'Exploration', '...","{'2689': 'Action Queue', '2839': 'Action Retri...","{'59218': 'Category: Dungeon Crawler', '67953'...","{'365186': 'The Crimson Scales', '367749': 'Th...","{'77084': 'Alexandr Elichev', '78961': 'Josh T...",{},{'390478': 'Gloomhaven (Second Edition)'},{'69802': 'Isaac Childres'},"{'27425': 'Cephalofair Games', '4304': 'Albi',...","{'295770': 'Frosthaven', '291457': 'Gloomhav

In [11]:
# Get existing ratings if continuing (prefer DuckDB snapshot if present)
existing_ratings = None
duckdb_path = data_dir / "ratings.duckdb"
if duckdb_path.exists():
    logger.info(f"Continuing from DuckDB ratings at: {duckdb_path}")
    con = duckdb.connect(str(duckdb_path))
    try:
        existing_ratings = build_wide_ratings_df_from_duckdb(con)
    finally:
        con.close()
else:
    ratings_files = list(data_dir.glob("boardgame_ratings_*.parquet"))
    if ratings_files:
        latest_ratings = max(ratings_files, key=lambda x: x.stat().st_mtime)
        logger.info(f"Continuing from ratings file: {latest_ratings}")
        existing_ratings = pd.read_parquet(latest_ratings)

2025-09-17 15:17:06,823 - __main__ - INFO - Continuing from DuckDB ratings at: /home/ubuntu/git/pax_tt_recommender/data/crawler/ratings.duckdb


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Walk through get_boardgame_ratings

In [ ]:
# get_boardgame_ratings(
#     boardgame_data=df_games,
#     boardgame_ratings=existing_ratings,
#     batch_saves=True,
#     drop_partial_ratings=args.drop_partial_ratings,
#     update_numratings=args.update_numratings,
# )

In [31]:
boardgame_data=df_games
boardgame_ratings=existing_ratings

In [10]:
query_time = int(time())
save_path = data_dir / f"boardgame_ratings_{query_time}.parquet"

In [11]:
# Initialize DuckDB persistent store for ratings
duckdb_path = data_dir / "ratings.duckdb"
con = duckdb.connect(str(duckdb_path))
con.execute(
    """
    CREATE TABLE IF NOT EXISTS boardgame_ratings (
        game_id BIGINT,
        rating_round DOUBLE,
        username TEXT
    );
    """
)

In [12]:
# Index to speed up de-dup checks
con.execute(
    "CREATE INDEX IF NOT EXISTS idx_boardgame_ratings ON boardgame_ratings(game_id, rating_round, username);"
)

In [32]:
boardgame_master_dict = {}
boardgame_data_ratings = boardgame_data.loc[boardgame_data["numratings"] > 100].sort_values(
    by="numratings", ascending=True
)
boardgame_ids = boardgame_data_ratings["id"].tolist()

In [33]:
print(df_games.shape)
print(boardgame_data_ratings.shape)

(131938, 37)
(16673, 37)


In [34]:
df_ratings_len = boardgame_ratings.copy()
df_ratings_len = df_ratings_len.drop(columns=["id"])
df_ratings_len = df_ratings_len.fillna("")
for col in df_ratings_len.columns:
    df_ratings_len[col] = df_ratings_len[col].apply(len)
df_ratings_pulled = pd.DataFrame(
    {
        "id": boardgame_ratings["id"].tolist(),
        "ratings_pulled": df_ratings_len.sum(axis=1).tolist(),
    }
)
boardgame_data_ratings = boardgame_data_ratings.merge(df_ratings_pulled, on="id", how="left")
completed_ids = boardgame_data_ratings.loc[
    (boardgame_data_ratings["ratings_pulled"] - boardgame_data_ratings["numratings"])
    / (boardgame_data_ratings["numratings"])
    >= -0.1,
    "id",
].tolist()
logger.info(
    f"Found {len(completed_ids)} boardgames with all ratings already pulled to completion"
)
df_missing_ratings = boardgame_data_ratings.loc[
    (boardgame_data_ratings["ratings_pulled"] - boardgame_data_ratings["numratings"])
    / (boardgame_data_ratings["numratings"])
    < -0.1
]
logger.info(
    f"Found {df_missing_ratings.shape[0]} boardgames with missing ratings"
)

2025-09-16 17:49:14,213 - __main__ - INFO - Found 15940 boardgames with all ratings already pulled to completion
2025-09-16 17:49:14,216 - __main__ - INFO - Found 733 boardgames with missing ratings


In [38]:
df_missing_ratings[["id","numratings", "ratings_pulled" ]].head()

,id,numratings,ratings_pulled
1537,233783,122,22
1550,4893,122,22
1552,130223,122,22
1570,411982,122,22
1578,298163,122,22


In [35]:
15940 +733

16673

In [17]:
boardgame_data_ratings.

(21684, 38)

# Export to parquet file

In [41]:
duckdb_path = data_dir / "ratings.duckdb"
con = duckdb.connect(str(duckdb_path))
df_ratings_wide = build_wide_ratings_df_from_duckdb(con)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [42]:
query_time = int(time())
save_path = data_dir / f"boardgame_ratings_{query_time}.parquet"

In [18]:
df_ratings_wide.dtypes

id       int64
0.0     object
0.5     object
1.0     object
1.5     object
2.0     object
2.5     object
3.0     object
3.5     object
4.0     object
4.5     object
5.0     object
5.5     object
6.0     object
6.5     object
7.0     object
7.5     object
8.0     object
8.5     object
9.0     object
9.5     object
10.0    object
dtype: object

In [38]:
df_ratings_wide_copy = df_ratings_wide.copy()
for col in df_ratings_wide_copy.columns:
    if col == "id":
        df_ratings_wide_copy[col] = df_ratings_wide_copy[col].astype(str)
    else:
        df_ratings_wide_copy[col] = df_ratings_wide_copy[col].apply(lambda x: x.tolist())

AttributeError: 'float' object has no attribute 'tolist'

In [39]:
for col in df_ratings_wide_nona.columns:
    if col == "id":
        df_ratings_wide_nona[col] = df_ratings_wide_nona[col].astype(str)
    else:
        df_ratings_wide_nona[col] = df_ratings_wide_nona[col].apply(lambda x: x.tolist())

AttributeError: 'str' object has no attribute 'tolist'

In [26]:
# df_ratings_wide_nona = df_ratings_wide.fillna(value='None')
df_ratings_wide_nona[["id", "10.0"]].head().to_parquet(save_path)

ValueError: Can't infer object conversion type: 0    [the life of games, jackcres, zremagem, ntroll...
1    [RedHawk, riddell, Mr Freedly, the_bummer, fla...
2    [Springfisher, Paul Gower, SteffenS, brax, sha...
3    [InnsmouthMedic, mdcjoshua, rdsmith, sissouvic...
4    [lstill, Haffner, lifesorceress, Menma, davest...
Name: 10.0, dtype: object

In [23]:
df_ratings_wide_nona.head()

,id,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,5.5,6.0,6.5,7.0,7.5,8.0,8.5,9.0,9.5,10.0
0,1,None,None,"[The Doctor, SilverDust, jsect, JBCon, cmason6...",None,"[ZiggyZambo, soulceress, clanche33, NoxMortem,...",[paraponzi],"[sourwyrm, Beowuulf1, hmmm, lmnop, Bridge83, F...","[heavydisking, RalfAMM, ColdFinger, Biku-Beku,...","[MULRAH, jmdsplotter, JimmyJingle, Legomancer,...","[bkleinwort, ChrB, pyaeen, denu2, Aurum, Paulo...","[shilinski, mrraow, jaystainbrook, DougOrleans...","[bcooperok, belyakoff, u8nogard, GameMasterX0,...","[dhoylman, brianeikunst, Csigs, ptper, StyxPar...","[counterfeit, happycamper, achilles1910, Jeter...","[Francisco Arena, aeng, rtrowan, n_and, yetano...","[madburner, Mikos, AntiGeek, beckerc, everlong...","[Temprament, ferralferrets, VolkoV, ThaisMunk,...","[mgringo, MikeyMike79, Kevin Rohrer, Wes Robii...","[dajial, JacksSpleen, Downblaw, Buzz, brainrob...","[Mycelia, babopiri, fendwick, ChrisHSLeach, Mi...","[the life of games, jackcres, zremagem, ntroll..."
1,2,None,None,"[Sheylon, Caillte, songjiexiong]",None,"[Bagherra, SnugglesVelociraptor]",None,"[BurkhartNicholas, w4terfall, lori6569, Mack T...",[KingsElite],"[Gola, himokl, Jeneki, talexdavis, Malachi, bo...",[brentont],"[Nightmare, Frax Beowor, TigerTailz, dingotrax...","[relfy, Estarriol, shirtman, Starsunsky, Eric ...","[Elf__Man, Rliyen, Windmilling, loopster70, gr...","[tonguepaste, rocksnrolls, CounterClockWise, M...","[NickBottom, DSeagraves, brbender, RandallPeek...","[jvburns, 1Mmirg, NiceguyAlex, gagster, latind...","[TRagedy, antaresecho, Transmuter, Fromage, ol...","[gameguru, Frankenfletch, Board Game Nutt]","[ElfShot, DJBarc, Sgt Trip, RichardIngram, mag...","[Mangle Paw, cguild]","[RedHawk, riddell, Mr Freedly, the_bummer, fla..."
2,3,None,None,"[Cnaldo, SilverDust, jamespisanich, monosolo, ...",None,"[Serious Fungamer, toukouy, jtj608, kirkusmaxi...",[bilbo20033],"[drdru82, capthowdy_00, blueatheart, schipa, x...","[TT Boy, Thatsnotme]","[aumi, hacksword, drakken, Vasquez21, grouchys...","[elmariachi, sisteray, j0ran, LoveGodZiming, a...","[Extreme, shilinski, DougOrleans, rorschah, Da...","[jarratt, trutu, jwallerrocks, cathyr19355, Zy...","[ShirKhan, Geese, onigame, Juxtatype, Stefan B...","[lighthousemay, morx, KMachi, prismofeverythin...","[GoTeamVenture, 3dicebombers, Smoky McAshtray,...","[aristid, opiumseed, joaoio, Lemon_pie, Morgan...","[RonStock, Bixby, caradoc, Naib_Stilgar, johan...","[matchbox1202, Majiken, Ggiron, lukasani, masa...","[shjivak, nocc32, pjly, zuzusdad, dwgteach, Ch...","[lukasaurus, CaseyP, cantimfelos, speavy, selu...","[Springfisher, Paul Gower, SteffenS, brax, sha..."
3,4,None,None,None,None,None,None,"[DSHStratRat2, Solipsiste, Chris Barnard, Chri...",None,"[Kosmonaut, shilinski, heli, barber, PfrBraun,...",[punkin312],"[matrix1970, gonzo104, shumyum, fsnam, tweetle...","[matteo_perin, mi_de, Gorgoneion, tallboy, Lan...","[puffinge, HRune, deltarn, Tatsu, SpieleHolger...","[cadawn, EYE of NiGHT, Tanayan, uribe, damaloc...","[Bryan Arroyo, Akke, Stirlingmoomoo, Corwyn59,...","[markus_mzh, ftola, bop517, IOIXIOI, Taggrip, ...","[PJefferies, pwotruba, Walt Mulder, Alan How, ...","[BohnanZar, echoota]","[arthemix, chuft, Bernard, nadle, mothertrucki...",None,"[InnsmouthMedic, mdcjoshua, rdsmith, sissouvic..."
4,5,None,None,"[Arontje, KAGE13, Abner Nirtzman, SilverDust, ...",[stormseraph],"[gordonplayer, ryanmaesen, CSoren, asteria, Co...",[lorry58],"[Lawrence Hung, Dreamgirl, ericgorr, beckerc, ...","[Wald, Agent Washingtub, McKell, B2010, duvalm...","[Scholle, joelee, Mijjy, Oktober, Darryl, Sagi...","[magnoguido, NaterPotater, Silverman, jdakaplu...","[Halloran, allanpanizzi, ETsc2, AntiGeek, sara...","[aceraxon, PontusNalle, boaz0norton, marioagui...","[ReptilianSamurai, bdemann, phinfan2003, Miste...","[shcheah, Louis XIX, Gibmaatsuki, preachain, b...","[danangbg, SourCreamSuicide, jaycel1, muratzki...","[wicuwe, Simon_Weinberg, nadzine, ektorakos, 1...","[LortGob, vmagiera, thegnobo, aliakba

In [47]:
df_ratings_wide.to_parquet(save_path)

In [23]:
con.close()

In [5]:
df_tmp = pd.read_parquet("../../data/crawler/boardgame_ratings_1758126429.parquet")
df_tmp.head()

,id,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,5.5,6.0,6.5,7.0,7.5,8.0,8.5,9.0,9.5,10.0
0,1,None,None,"[The Doctor, SilverDust, jsect, JBCon, cmason6...",None,"[ZiggyZambo, soulceress, clanche33, NoxMortem,...",[paraponzi],"[sourwyrm, Beowuulf1, hmmm, lmnop, Bridge83, F...","[heavydisking, RalfAMM, ColdFinger, Biku-Beku,...","[MULRAH, jmdsplotter, JimmyJingle, Legomancer,...","[bkleinwort, ChrB, pyaeen, denu2, Aurum, Paulo...","[shilinski, mrraow, jaystainbrook, DougOrleans...","[bcooperok, belyakoff, u8nogard, GameMasterX0,...","[dhoylman, brianeikunst, Csigs, ptper, StyxPar...","[counterfeit, happycamper, achilles1910, Jeter...","[Francisco Arena, aeng, rtrowan, n_and, yetano...","[madburner, Mikos, AntiGeek, beckerc, everlong...","[Temprament, ferralferrets, VolkoV, ThaisMunk,...","[mgringo, MikeyMike79, Kevin Rohrer, Wes Robii...","[dajial, JacksSpleen, Downblaw, Buzz, brainrob...","[Mycelia, babopiri, fendwick, ChrisHSLeach, Mi...","[the life of games, jackcres, zremagem, ntroll..."
1,2,None,None,"[Sheylon, Caillte, songjiexiong]",None,"[Bagherra, SnugglesVelociraptor]",None,"[BurkhartNicholas, w4terfall, lori6569, Mack T...",[KingsElite],"[Gola, himokl, Jeneki, talexdavis, Malachi, bo...",[brentont],"[Nightmare, Frax Beowor, TigerTailz, dingotrax...","[relfy, Estarriol, shirtman, Starsunsky, Eric ...","[Elf__Man, Rliyen, Windmilling, loopster70, gr...","[tonguepaste, rocksnrolls, CounterClockWise, M...","[NickBottom, DSeagraves, brbender, RandallPeek...","[jvburns, 1Mmirg, NiceguyAlex, gagster, latind...","[TRagedy, antaresecho, Transmuter, Fromage, ol...","[gameguru, Frankenfletch, Board Game Nutt]","[ElfShot, DJBarc, Sgt Trip, RichardIngram, mag...","[Mangle Paw, cguild]","[RedHawk, riddell, Mr Freedly, the_bummer, fla..."
2,3,None,None,"[Cnaldo, SilverDust, jamespisanich, monosolo, ...",None,"[Serious Fungamer, toukouy, jtj608, kirkusmaxi...",[bilbo20033],"[drdru82, capthowdy_00, blueatheart, schipa, x...","[TT Boy, Thatsnotme]","[aumi, hacksword, drakken, Vasquez21, grouchys...","[elmariachi, sisteray, j0ran, LoveGodZiming, a...","[Extreme, shilinski, DougOrleans, rorschah, Da...","[jarratt, trutu, jwallerrocks, cathyr19355, Zy...","[ShirKhan, Geese, onigame, Juxtatype, Stefan B...","[lighthousemay, morx, KMachi, prismofeverythin...","[Gogolski, wich, AmassGames, cheddarhead52, pr...","[aristid, opiumseed, joaoio, Lemon_pie, Morgan...","[Arnaldo, eraserhead, SmartChimp, thedacker, j...","[matchbox1202, Majiken, Ggiron, lukasani, masa...","[shjivak, nocc32, pjly, zuzusdad, dwgteach, Ch...","[lukasaurus, CaseyP, cantimfelos, speavy, selu...","[Springfisher, Paul Gower, SteffenS, brax, sha..."
3,4,None,None,None,None,None,None,"[DSHStratRat2, Solipsiste, Chris Barnard, Chri...",None,"[Kosmonaut, shilinski, heli, barber, PfrBraun,...",[punkin312],"[matrix1970, gonzo104, shumyum, fsnam, tweetle...","[matteo_perin, mi_de, Gorgoneion, tallboy, Lan...","[puffinge, HRune, deltarn, Tatsu, SpieleHolger...","[cadawn, EYE of NiGHT, Tanayan, uribe, damaloc...","[Bryan Arroyo, Akke, Stirlingmoomoo, Corwyn59,...","[markus_mzh, ftola, bop517, IOIXIOI, Taggrip, ...","[PJefferies, pwotruba, Walt Mulder, Alan How, ...","[BohnanZar, echoota]","[arthemix, chuft, Bernard, nadle, mothertrucki...",None,"[InnsmouthMedic, mdcjoshua, rdsmith, sissouvic..."
4,5,None,None,"[Arontje, KAGE13, Abner Nirtzman, SilverDust, ...",[stormseraph],"[gordonplayer, ryanmaesen, CSoren, asteria, Co...",[lorry58],"[Lawrence Hung, Dreamgirl, ericgorr, beckerc, ...","[Wald, Agent Washingtub, McKell, B2010, duvalm...","[Scholle, joelee, Mijjy, Oktober, Darryl, Sagi...","[magnoguido, NaterPotater, Silverman, jdakaplu...","[Halloran, allanpanizzi, ETsc2, AntiGeek, sara...","[aceraxon, PontusNalle, boaz0norton, marioagui...","[cgoodrick, Barrow-Wight, Brave Sir Robin, Jas...","[shcheah, Louis XIX, Gibmaatsuki, preachain, b...","[rjstreet, Noaros, TaijiJohn, numage, Solamar,...","[wicuwe, Simon_Weinberg, nadzine, ektorakos, 1...","[LortGob, vmagiera, thegnobo, aliakba

In [45]:
df_tmp["id"].iloc[0]

np.int64(13951)

In [34]:
type(df_ratings_wide_nona["10.0"].iloc[0])

numpy.ndarray